# nested-param-group-loop — worked example 3: manual zero_grad via the nested loop

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `nested-param-group-loop`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Not just the step, but `zero_grad` also uses the outer-over-groups, inner-over-params double loop. The recommended reset is `p.grad = None` (set-to-none), which lets the gradient tensor be garbage collected and matches `optimizer.zero_grad(set_to_none=True)`.

## Worked solution

We build a two-group optimizer and populate gradients on every parameter. To reimplement `zero_grad`, the outer loop walks `param_groups` and the inner loop walks `group['params']`, setting `p.grad = None` for each. There are no hyperparameters to read here, but the nested structure is identical to the step loop, which is the point: the same traversal pattern serves both operations. We seed, confirm gradients exist before, run the manual zero, and print that every parameter's `.grad` is now `None`.

In [ ]:
import torch as t

t.manual_seed(2)

p1 = t.nn.Parameter(t.randn(3))
p2 = t.nn.Parameter(t.randn(3))
p3 = t.nn.Parameter(t.randn(3))
opt = t.optim.SGD([
    {'params': [p1, p2], 'lr': 0.1},
    {'params': [p3], 'lr': 0.3},
])
for p in (p1, p2, p3):
    p.grad = t.randn(3)

def manual_zero_grad(optimizer):
    for group in optimizer.param_groups:
        for p in group['params']:
            p.grad = None

manual_zero_grad(opt)
print('all grads None:', all(p.grad is None for p in (p1, p2, p3)))